# Predicting DVD Rental Durations 📀
**Author:** Francisco

## Project Overview
A DVD rental company wants to figure out how many days a customer will rent a DVD for based on several features. The goal of this project is to build a regression model that predicts this rental duration and achieves a Mean Squared Error (MSE) of 3 or less on a test set. This will help the company with efficient inventory planning.

![dvd_image](dvd_image.jpg)

A DVD rental company needs your help! They want to figure out how many days a customer will rent a DVD for based on some features and has approached you for help. They want you to try out some regression models which will help predict the number of days a customer will rent a DVD for. The company wants a model which yeilds a MSE of 3 or less on a test set. The model you make will help the company become more efficient inventory planning.

The data they provided is in the csv file `rental_info.csv`. It has the following features:
- `"rental_date"`: The date (and time) the customer rents the DVD.
- `"return_date"`: The date (and time) the customer returns the DVD.
- `"amount"`: The amount paid by the customer for renting the DVD.
- `"amount_2"`: The square of `"amount"`.
- `"rental_rate"`: The rate at which the DVD is rented for.
- `"rental_rate_2"`: The square of `"rental_rate"`.
- `"release_year"`: The year the movie being rented was released.
- `"length"`: Lenght of the movie being rented, in minuites.
- `"length_2"`: The square of `"length"`.
- `"replacement_cost"`: The amount it will cost the company to replace the DVD.
- `"special_features"`: Any special features, for example trailers/deleted scenes that the DVD also has.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"`: These columns are dummy variables of the rating of the movie. It takes the value 1 if the move is rated as the column name and 0 otherwise. For your convinience, the reference dummy has already been dropped.

In [11]:
# Import fundamental libraries
import pandas as pd
import numpy as np

# Import machine learning tools
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Import specific models and tools for the pipeline
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

## 1. Data Preprocessing & Feature Engineering
First, we need to load the data and create our target variable (`rental_length_days`). 
We will calculate the difference between the return date and the rental date. After that, we need to extract information from the `special_features` column by creating dummy variables for "Deleted Scenes" and "Behind the Scenes".

In [12]:
# Read in the dataset
df_rental = pd.read_csv("rental_info.csv")

# Create the target variable: rental duration in days
df_rental["rental_length"] = pd.to_datetime(df_rental["return_date"]) - pd.to_datetime(df_rental["rental_date"])
df_rental["rental_length_days"] = df_rental["rental_length"].dt.days

# Feature Engineering: Create dummy variables from 'special_features'
# np.where works like an IF statement: if condition is true, return 1, else 0.
df_rental["deleted_scenes"] = np.where(df_rental["special_features"].str.contains("Deleted Scenes"), 1, 0)
df_rental["behind_the_scenes"] = np.where(df_rental["special_features"].str.contains("Behind the Scenes"), 1, 0)

# Display the first few rows to verify changes
df_rental.head()

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2,rental_length,rental_length_days,deleted_scenes,behind_the_scenes
0,2005-05-25 02:54:33+00:00,2005-05-28 23:40:33+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,3 days 20:46:00,3,0,1
1,2005-06-15 23:19:16+00:00,2005-06-18 19:24:16+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,2 days 20:05:00,2,0,1
2,2005-07-10 04:27:45+00:00,2005-07-17 10:11:45+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,7 days 05:44:00,7,0,1
3,2005-07-31 12:06:41+00:00,2005-08-02 14:30:41+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,2 days 02:24:00,2,0,1
4,2005-08-19 12:30:04+00:00,2005-08-23 13:35:04+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,4 days 01:05:00,4,0,1


## 2. Data Splitting and Leakage Prevention
To train our models properly, we must separate our features ($X$) from our target variable ($y$). 
Crucially, we must drop any columns that "leak" information about the target variable (like the original dates and the exact time delta), as well as the text column we already processed. Then, we split the data into training (80%) and testing (20%) sets.

In [13]:
# Define columns that leak data or are no longer needed
cols_to_drop = ["special_features", "rental_length", "rental_length_days", "rental_date", "return_date"]

# Split into feature matrix (X) and target vector (y)
X = df_rental.drop(cols_to_drop, axis=1)
y = df_rental["rental_length_days"]

# Split into training and test sets (20% for testing)
# random_state=9 ensures reproducibility as requested
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    random_state=9)

## 3. Feature Selection with Lasso
Before running a standard linear regression, we will use a Lasso Regression model. 
Lasso has a regularization property that shrinks the coefficients of less important features to exactly zero. We can use this property to select only the most relevant features for our final linear model.

In [14]:
# Initialize the Lasso model
lasso = Lasso(alpha=0.3, random_state=9) 

# Train the model to find the feature coefficients
lasso.fit(X_train, y_train)
lasso_coef = lasso.coef_

# Perform feature selection: Keep only columns where the coefficient is strictly greater than 0
X_lasso_train = X_train.iloc[:, lasso_coef > 0]
X_lasso_test = X_test.iloc[:, lasso_coef > 0]

print(f"Features reduced from {X_train.shape[1]} to {X_lasso_train.shape[1]}")

Features reduced from 14 to 3


## 4. Modeling Pipeline
Now we will test two different approaches to see which one performs better:
1. **Ordinary Least Squares (OLS):** A standard linear regression using only the features selected by Lasso.
2. **Random Forest Regressor:** A powerful ensemble tree model. We will use `RandomizedSearchCV` to tune its hyperparameters and find the optimal configuration.

In [15]:
# --- Approach 1: OLS with Lasso-selected features ---
ols = LinearRegression()
ols.fit(X_lasso_train, y_train)

# Predict and calculate MSE
y_test_pred = ols.predict(X_lasso_test)
mse_lin_reg_lasso = mean_squared_error(y_test, y_test_pred)

print(f"Linear Regression (with Lasso selection) MSE: {mse_lin_reg_lasso:.4f}")

# --- Approach 2: Random Forest with Hyperparameter Tuning ---
# Define the hyperparameter grid to search
param_dist = {
    'n_estimators': np.arange(1, 101, 1), # Number of trees
    'max_depth': np.arange(1, 11, 1)      # Maximum depth of trees
}

rf = RandomForestRegressor()

# Set up the random search with cross-validation
rand_search = RandomizedSearchCV(rf, 
                                 param_distributions=param_dist, 
                                 cv=5, 
                                 random_state=9)

# Fit the random search object to find the best parameters
rand_search.fit(X_train, y_train)
hyper_params = rand_search.best_params_
print(f"Best Random Forest parameters found: {hyper_params}")

# Train the final Random Forest model with the best parameters
rf_best = RandomForestRegressor(n_estimators=hyper_params["n_estimators"], 
                                max_depth=hyper_params["max_depth"], 
                                random_state=9)
rf_best.fit(X_train, y_train)

# Predict and calculate MSE
rf_pred = rf_best.predict(X_test)
mse_random_forest = mean_squared_error(y_test, rf_pred)

print(f"Random Forest MSE: {mse_random_forest:.4f}")

Linear Regression (with Lasso selection) MSE: 4.8123
Best Random Forest parameters found: {'n_estimators': 51, 'max_depth': 10}
Random Forest MSE: 2.2257


## 5. Final Model Selection
The final step is to evaluate which model achieved an MSE of less than 3 and performed the best overall. We will save this as our `best_model`.

In [16]:
# The Random Forest provides the lowest MSE and meets the < 3 requirement
best_model = rf_best
best_mse = mse_random_forest

print(f"The best model is a Random Forest with an MSE of {best_mse:.4f} on the test set.")

The best model is a Random Forest with an MSE of 2.2257 on the test set.
